In [2]:
import pandas as pd
import json
from api_caller import call_api
import re

In [3]:
df = pd.read_csv("data/EU Debates/preprocessed/filtered.csv")

In [4]:
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError("No JSON found")

def classify_batch(texts):
    user_prompt = f"""You are a political ideology detection system.

Your task is to detect STRONG ideological value expression.

A speech should receive a high score ONLY IF it clearly expresses
a political principle or ideological position that could be mapped
onto a political survey (e.g., redistribution, EU integration,
national sovereignty, migration, social equality, market regulation,
democracy, rule of law, etc.).

STRICT CRITERIA:

The speech MUST:
- Advocate or oppose a political principle
- Express how society, the EU, or government SHOULD be structured
- Reveal a stable ideological commitment attributable to the speaker or their party

The speech must NOT be classified as value-expressing if it:
- Expresses generic hope or praise
- Uses polite or diplomatic language
- Evaluates events without ideological reasoning
- Contains general positive or negative sentiment only

IMPORTANT:
Generic approval is NOT ideological.

Be extremely conservative.
If the speech does not clearly state a political principle,
assign a score below 0.2.

Text:
{texts}

Scoring guide:
0.0-0.2 → no ideological value content
0.3-0.5 → weak or vague value signals
0.6-0.8 → clear ideological positioning
0.9-1.0 → strong, explicit ideological commitment

Return STRICT JSON:
{{
  "score": 0.0-1.0,
  "reason": "..."
}}

IMPORTANT: Return valid JSON, even if the score is 0.0.
"""
    response = None
    while True:
      try:
        response = call_api(user_prompt)
        content = response["choices"][0]["message"]["content"]
        return extract_json(content)
      except Exception as e:
        print(f"{e} thrown for {texts}, {response}")
      except:
        print(f"Failed for other reason on {texts}, repeating")


In [5]:
def process_df(df, ifrom, istep=500, batch_size=1):
    j = ifrom
    while j < len(df) - istep:
        try:
            df_subset = df.iloc[j:j+istep].copy()
            indices = list(range(0, len(df_subset), batch_size))
            results = []

            for i in range(0, len(df_subset), batch_size):
                print(f"Progress: {(j+i)/len(df)*100}%")
                batch_texts = df_subset["text"].iloc[i:i+batch_size].tolist()[0]
                batch_results = []
                batch_results = classify_batch(batch_texts)
                results.append(batch_results)
            
            df_subset["reason"] = [r["reason"] for r in results]
            df_subset["score"] = [r["score"] for r in results]
            final_df = df_subset[["text", "score", "reason", "speaker_party"]]
            final_df.to_csv(
                f"data/EU Debates/scored_data_{j}_{j+istep}.csv",
                sep=";",
                index=False,
                encoding="utf-8-sig"
            )

            j += istep
        except Exception as e:
            print(e)

    df_subset = df.iloc[j:].copy()
    indices = list(range(0, len(df_subset), batch_size))
    results = []

    for i in range(0, len(df_subset), batch_size):
        print(f"Progress: {(j+i)/len(df)*100}%")
        batch_texts = df_subset["text"].iloc[i:i+batch_size].tolist()
        batch_results = classify_batch(batch_texts)
        results.extend(batch_results)
    
    df_subset["reason"] = [r["reason"] for r in results]
    df_subset["score"] = [r["score"] for r in results]
    final_df = df_subset[["text", "score", "reason", "soeaker_party"]]
    final_df.to_csv(
        f"data/EU Debates/scored_data_{j}_{len(df)}.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

In [ ]:
process_df(df, ifrom=9500)

Progress: 11.577741487313233%
Progress: 11.578960196943475%
Progress: 11.58017890657372%
Progress: 11.581397616203963%
Progress: 11.582616325834207%
Progress: 11.58383503546445%
Progress: 11.585053745094694%
Progress: 11.586272454724938%
Progress: 11.58749116435518%
Progress: 11.588709873985424%
Progress: 11.589928583615668%
Progress: 11.59114729324591%
Progress: 11.592366002876155%
Progress: 11.593584712506399%
Progress: 11.594803422136641%
Progress: 11.596022131766885%
Progress: 11.597240841397129%
Progress: 11.598459551027371%
Progress: 11.599678260657615%
Progress: 11.60089697028786%
Progress: 11.602115679918104%
Progress: 11.603334389548346%
Progress: 11.60455309917859%
Progress: 11.605771808808834%
Progress: 11.606990518439076%
Progress: 11.60820922806932%
Progress: 11.609427937699564%
Progress: 11.610646647329807%
Progress: 11.61186535696005%
Progress: 11.613084066590295%
Progress: 11.614302776220537%
Progress: 11.615521485850781%
Progress: 11.616740195481025%
Progress: 11.61795